# RECIP-E — Interpretador (front-end) em Scheme

Projeto da disciplina **MC346 — Paradigmas de Programação** (UNICAMP).

Este é o **notebook único** do grupo: todas as etapas do processamento da DSL
**RECIP-E** vivem aqui dentro, em ordem de pipeline — léxico → sintático →
impressão/validação. Kernel: **Calysto Scheme**.

> **Estado: Dia 2 (Dom 28/06)** — a seção **1. Lexer** (Pessoa A) está completa,
> incluindo agora a **indentação** (`NEWLINE` / `INDENT` / `DEDENT`). As seções
> **2. Parser** (Pessoa B) e **3. Pretty-print + Validação** (Pessoa C) estão
> reservadas.
>
> Para conferir tudo: **Kernel → Restart & Run All**.

## 0. Setup / Contrato do token

Tudo no projeto conversa por meio de **tokens**. Combinamos um formato simples:

```
token = (tipo lexema linha)
```

- **tipo** — um símbolo: `NUMBER STRING IDENT KEYWORD VERB UNIT PREP PONT` e os
  tokens de estrutura `NEWLINE INDENT DEDENT` (gerados pela indentação).
- **lexema** — o texto original (string); vazio `""` para `NEWLINE/INDENT/DEDENT`.
- **linha** — número da linha no código-fonte (útil para mensagens de erro).

As funções `faz-token`, `tipo-tok`, `lexema-tok`, `linha-tok` são a **interface**
que a Pessoa B (parser) vai usar para ler os tokens.

In [ ]:
;; ---- Contrato do token: (tipo lexema linha) ----
;; tipos possiveis:
;;   NUMBER STRING IDENT KEYWORD VERB UNIT PREP PONT   (conteudo)
;;   NEWLINE INDENT DEDENT                             (estrutura/indentacao)
(define (faz-token tipo lexema linha) (list tipo lexema linha))
(define (tipo-tok   t) (car   t))
(define (lexema-tok t) (cadr  t))
(define (linha-tok  t) (caddr t))

;; ---- Tabelas de palavras reservadas da linguagem ----
(define palavras-bloco
  (list "RECEITA" "INGREDIENTES" "UTENSILIOS" "METODOS"))

(define palavras-controle
  (list "ESTE" "é" "AGUARDAR" "VERIFICAR" "SE" "ESTA"
        "CONTINUAR" "SENAO" "AO_MESMO_TEMPO"
        "GRAUS" "MINUTOS" "HORAS" "SEGUNDOS"))

(define preposicoes
  (list "EM" "COM" "USANDO" "A" "POR" "ATE" "NO_ESTILO"))

(define verbos
  (list "ADICIONAR" "MISTURAR" "BATER" "TEMPERAR" "ASSAR" "FRITAR"
        "COZINHAR" "REFOGAR" "GRELHAR" "CORTAR" "RALAR" "ESCORRER"
        "COLOCAR" "DECORAR"))

(define unidades
  (list "g" "kg" "ml" "l" "unidade" "colher" "colher_cha"
        "xcara" "pitada" "a_gosto" "faca"))

;; verifica se uma palavra esta numa lista (devolve #t ou #f)
(define (esta-em? palavra lista)
  (if (member palavra lista) #t #f))

;; descobre o tipo de uma palavra ja lida pelo lexer
(define (classifica palavra)
  (cond
    ((esta-em? palavra palavras-bloco)    'KEYWORD)
    ((esta-em? palavra palavras-controle) 'KEYWORD)
    ((esta-em? palavra preposicoes)       'PREP)
    ((esta-em? palavra verbos)            'VERB)
    ((esta-em? palavra unidades)          'UNIT)
    (else                                 'IDENT)))

## 1. Lexer (Pessoa A)

O lexer transforma o texto da receita numa **lista de tokens**. Ele reconhece
números, strings, identificadores e palavras reservadas, ignora comentários
(`//` e `/* */`) e — desde o Dia 2 — entende a **indentação**, emitindo
`NEWLINE` no fim de cada linha e `INDENT`/`DEDENT` quando o recuo aumenta ou
diminui.

### 1.1 Predicados auxiliares

Funções pequenas para classificar caracteres. `TAB` e `RET` são definidos via
`integer->char` para não depender de literais de caractere especiais.

In [ ]:
;; alguns caracteres "invisiveis"
(define TAB (integer->char 9))    ; tabulacao
(define RET (integer->char 13))   ; carriage return

;; e um digito de 0 a 9?
(define (eh-digito? c)
  (and (char>=? c #\0) (char<=? c #\9)))

;; e uma letra (A-Z, a-z), underline ou letra acentuada (é, ç, ...)?
(define (eh-letra? c)
  (let ((code (char->integer c)))
    (or (and (>= code 65) (<= code 90))     ; A-Z
        (and (>= code 97) (<= code 122))    ; a-z
        (= code 95)                         ; _
        (>= code 128))))                    ; acentuadas

;; e um espaco em branco (sem contar a quebra de linha)?
(define (eh-espaco? c)
  (or (char=? c #\space) (char=? c TAB) (char=? c RET)))

### 1.2 Remoção de comentários

RECIP-E aceita comentário de linha `// ...` e de bloco `/* ... */`. Antes de
tokenizar, trocamos os comentários por nada — **mas preservamos as quebras de
linha**, para o número de linha continuar certo — e tomamos cuidado para não
apagar um `//` que esteja **dentro de uma string**.

In [ ]:
;; remove comentarios // e /* */, preservando quebras de linha e strings
(define (remove-comentarios s)
  (let ((n (string-length s)))
    (define (loop i estado acc)
      (if (>= i n)
          (list->string (reverse acc))
          (let ((c (string-ref s i)))
            (cond
              ;; ----- texto normal -----
              ((eq? estado 'normal)
               (cond
                 ((char=? c #\")
                  (loop (+ i 1) 'string (cons c acc)))
                 ((and (char=? c #\/) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\/))
                  (loop (+ i 2) 'linha acc))
                 ((and (char=? c #\/) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\*))
                  (loop (+ i 2) 'bloco acc))
                 (else
                  (loop (+ i 1) 'normal (cons c acc)))))
              ;; ----- dentro de uma string -----
              ((eq? estado 'string)
               (if (char=? c #\")
                   (loop (+ i 1) 'normal (cons c acc))
                   (loop (+ i 1) 'string (cons c acc))))
              ;; ----- comentario de linha: pula ate a quebra -----
              ((eq? estado 'linha)
               (if (char=? c #\newline)
                   (loop (+ i 1) 'normal (cons c acc))
                   (loop (+ i 1) 'linha acc)))
              ;; ----- comentario de bloco: pula ate */ -----
              ((eq? estado 'bloco)
               (cond
                 ((and (char=? c #\*) (< (+ i 1) n)
                       (char=? (string-ref s (+ i 1)) #\/))
                  (loop (+ i 2) 'normal acc))
                 ((char=? c #\newline)
                  (loop (+ i 1) 'bloco (cons c acc)))   ; mantem a linha
                 (else
                  (loop (+ i 1) 'bloco acc))))
              (else
               (loop (+ i 1) 'normal acc))))))
    (loop 0 'normal '())))

### 1.3 Leitura de números, identificadores e strings

Cada leitor recebe a string `s`, a posição inicial `i` e um limite `fim`, e
devolve a **posição final** do token (onde ele termina).

In [ ]:
;; le um numero (inteiro ou decimal); devolve a posicao final
(define (le-numero s i fim)
  (cond
    ((>= i fim) i)
    ((eh-digito? (string-ref s i)) (le-numero s (+ i 1) fim))
    ((and (char=? (string-ref s i) #\.) (< (+ i 1) fim)
          (eh-digito? (string-ref s (+ i 1))))
     (le-numero s (+ i 2) fim))
    (else i)))

;; le um identificador (letras, digitos, underline); devolve a posicao final
(define (le-ident s i fim)
  (if (and (< i fim)
           (or (eh-letra? (string-ref s i))
               (eh-digito? (string-ref s i))))
      (le-ident s (+ i 1) fim)
      i))

;; le uma string entre aspas; i aponta DEPOIS da aspa de abertura.
;; devolve a posicao logo apos a aspa de fechamento
(define (le-string s i fim)
  (cond
    ((>= i fim) fim)
    ((char=? (string-ref s i) #\") (+ i 1))
    (else (le-string s (+ i 1) fim))))

### 1.4 Indentação: `NEWLINE`, `INDENT`, `DEDENT`

Em RECIP-E o **recuo define o escopo** (não há chaves). O lexer processa o texto
**linha a linha** mantendo uma *pilha de níveis de indentação* (em número de
espaços; `tab` vale 4):

- recuo **maior** que o topo da pilha → empilha e emite **`INDENT`**;
- recuo **menor** → desempilha emitindo um **`DEDENT`** por nível fechado;
- recuo **igual** → nada;
- fim de cada linha com conteúdo → **`NEWLINE`**;
- linhas em branco (ou que só tinham comentário) são **ignoradas**;
- no fim do arquivo, fecha os níveis abertos com `DEDENT`.

> **Observação:** o arquivo deve usar recuo **consistente** (múltiplos de 4
> espaços). O alinhamento estético com espaços variáveis dos exemplos do PDF
> (ex.: números à direita) não é suportado nesta versão.

In [ ]:
;; indice da proxima quebra de linha a partir de i (ou n se nao houver)
(define (fim-da-linha s i n)
  (cond ((>= i n) n)
        ((char=? (string-ref s i) #\newline) i)
        (else (fim-da-linha s (+ i 1) n))))

;; primeiro indice >= i que NAO e espaco/tab (sem passar de fim)
(define (pula-brancos s i fim)
  (if (and (< i fim) (eh-espaco? (string-ref s i)))
      (pula-brancos s (+ i 1) fim)
      i))

;; "peso" da indentacao a partir de i: espaco = 1, tab = 4; para no 1o nao-branco
(define (peso-indentacao s i fim)
  (if (>= i fim)
      0
      (let ((c (string-ref s i)))
        (cond ((char=? c #\space) (+ 1 (peso-indentacao s (+ i 1) fim)))
              ((char=? c TAB)     (+ 4 (peso-indentacao s (+ i 1) fim)))
              (else 0)))))

;; ajusta a pilha para o novo nivel, gerando INDENT/DEDENT.
;; devolve um par (nova-pilha . novo-acc)
(define (ajusta-indent nivel pilha linha acc)
  (let ((topo (car pilha)))
    (cond
      ((> nivel topo)
       (cons (cons nivel pilha)
             (cons (faz-token 'INDENT "" linha) acc)))
      ((< nivel topo)
       (ajusta-indent nivel (cdr pilha) linha
                      (cons (faz-token 'DEDENT "" linha) acc)))
      (else
       (cons pilha acc)))))

;; no fim do arquivo, emite um DEDENT para cada nivel acima do nivel base (0)
(define (fecha-todos pilha linha acc)
  (if (or (null? pilha) (= (car pilha) 0))
      acc
      (fecha-todos (cdr pilha) linha
                   (cons (faz-token 'DEDENT "" linha) acc))))

### 1.5 A função `tokeniza`

`tokens-da-linha` quebra o conteúdo de **uma** linha em tokens; `tokeniza` junta
tudo: remove comentários, percorre linha a linha cuidando da indentação e
devolve a lista final de tokens. `mostra-tokens` só ajuda a visualizar.

In [ ]:
;; tokeniza o conteudo de UMA linha, do indice a ate b (exclusivo)
(define (tokens-da-linha s a b linha acc)
  (if (>= a b)
      acc
      (let ((c (string-ref s a)))
        (cond
          ((eh-espaco? c)
           (tokens-da-linha s (+ a 1) b linha acc))
          ((eh-digito? c)
           (let ((j (le-numero s a b)))
             (tokens-da-linha s j b linha
               (cons (faz-token 'NUMBER (substring s a j) linha) acc))))
          ((char=? c #\")
           (let ((j (le-string s (+ a 1) b)))
             (tokens-da-linha s j b linha
               (cons (faz-token 'STRING (substring s (+ a 1) (- j 1)) linha) acc))))
          ((eh-letra? c)
           (let* ((j (le-ident s a b))
                  (palavra (substring s a j)))
             (tokens-da-linha s j b linha
               (cons (faz-token (classifica palavra) palavra linha) acc))))
          ((char=? c #\,)
           (tokens-da-linha s (+ a 1) b linha (cons (faz-token 'PONT "," linha) acc)))
          ((char=? c #\:)
           (tokens-da-linha s (+ a 1) b linha (cons (faz-token 'PONT ":" linha) acc)))
          (else
           (tokens-da-linha s (+ a 1) b linha acc))))))

;; transforma o texto-fonte na lista de tokens (com NEWLINE/INDENT/DEDENT)
(define (tokeniza fonte)
  (let* ((s (remove-comentarios fonte))
         (n (string-length s)))
    (define (loop i linha pilha acc)
      (if (>= i n)
          (reverse (fecha-todos pilha linha acc))
          (let* ((fl  (fim-da-linha s i n))      ; fim da linha (quebra ou n)
                 (ini (pula-brancos s i fl)))     ; inicio do conteudo
            (if (>= ini fl)
                ;; linha em branco (ou so comentario): ignora
                (loop (+ fl 1) (+ linha 1) pilha acc)
                ;; linha com conteudo
                (let* ((nivel (peso-indentacao s i fl))
                       (par   (ajusta-indent nivel pilha linha acc))
                       (pilha2 (car par))
                       (acc2   (cdr par))
                       (acc3   (tokens-da-linha s ini fl linha acc2))
                       (acc4   (cons (faz-token 'NEWLINE "" linha) acc3)))
                  (loop (+ fl 1) (+ linha 1) pilha2 acc4))))))
    (loop 0 1 (list 0) '())))

;; mostra os tokens, um por linha (so para visualizar)
(define (mostra-tokens tokens)
  (for-each
    (lambda (t)
      (display (tipo-tok t))   (display "  ")
      (display (lexema-tok t)) (display "   (linha ")
      (display (linha-tok t))  (display ")")
      (newline))
    tokens))

### 1.6 Testes manuais

In [ ]:
;; uma linha simples: o comentario some e sobra um NEWLINE no fim
(mostra-tokens (tokeniza "10 g sal // isto e um comentario"))

In [ ]:
;; metadados com string:  NOME -> IDENT, : -> PONT, "..." -> STRING
(mostra-tokens (tokeniza "NOME: \"OmeleteComQueijo\""))

In [ ]:
;; comentario de bloco ocupando linhas inteiras: elas somem,
;; e a numeracao de linha do que sobra continua correta (linha 3)
(mostra-tokens
  (tokeniza "/* receita
   de teste */
ADICIONAR ovo EM tigela"))

In [ ]:
;; indentacao aninhada: repare nos INDENT/DEDENT e NEWLINE
(mostra-tokens
  (tokeniza "METODOS
    ADICIONAR ovo EM tigela
    AO_MESMO_TEMPO
        ADICIONAR oleo EM frigideira
        AGUARDAR 1 MINUTOS
    COLOCAR ovo EM prato"))

## 2. Parser (Pessoa B — a fazer)

Recebe a lista de tokens de `tokeniza` e constrói a **AST** (árvore sintática),
seguindo a gramática `G = (V, Σ, P, S)` da especificação. Usa a interface
`tipo-tok` / `lexema-tok` / `linha-tok` da seção 0, e os tokens de estrutura
`NEWLINE` / `INDENT` / `DEDENT` para delimitar blocos.

_Reservado para a Pessoa B._

## 3. Pretty-print + Validação (Pessoa C — a fazer)

Imprime a AST de forma legível e roda as verificações estáticas (declarar antes
de usar, soma de `USANDO` vs estoque, conflito de utensílio em `AO_MESMO_TEMPO`,
referência de sub-receita).

_Reservado para a Pessoa C._

## 4. Exemplos

### 4.1 Bloco INGREDIENTES

In [ ]:
;; sub-receita vem como STRING; repare no INDENT/DEDENT do bloco
(define exemplo-ingredientes
  "INGREDIENTES
    2 unidade ovo
    200 ml leite
    1 unidade \"CaldaDeChocolate\"")

(mostra-tokens (tokeniza exemplo-ingredientes))

### 4.2 Receita completa — Omelete com Queijo (seção 15)

Exemplo integrando metadados, `ESTE é`, `USANDO`, `AO_MESMO_TEMPO`,
`VERIFICAR SE/SENAO` e sub-receita. Indentação normalizada para múltiplos de 4
espaços (ver observação na seção 1.4).

In [ ]:
;; Omelete com Queijo — os comentarios // sao removidos pelo lexer
(define omelete
  "RECEITA
    NOME: \"OmeleteComQueijo\"
    PORCOES: 1
    NIVEL: FACIL
    TEMPO_PREPARO: 10 MINUTOS

INGREDIENTES
    2 unidade ovo
    20 g sal
    5 g pimenta_do_reino
    1 colher oleo
    1 unidade \"RecheioDeQueijo\"

UTENSILIOS
    1 frigideira
    1 garfo
    1 tigela
    1 prato

METODOS
    ADICIONAR ovo, sal USANDO 5 g EM tigela
    MISTURAR ovo, pimenta_do_reino USANDO 2 g EM tigela COM garfo
    ESTE é MISTURA01

    AO_MESMO_TEMPO
        ADICIONAR oleo EM frigideira
        AGUARDAR 1 MINUTOS

    ASSAR MISTURA01 EM frigideira A FOGO_MEDIO ATE FIRMAR

    VERIFICAR SE MISTURA01 ESTA FIRME
        CONTINUAR
    SENAO
        ASSAR MISTURA01 EM frigideira POR 2 MINUTOS

    ADICIONAR RecheioDeQueijo EM MISTURA01
    COLOCAR MISTURA01 EM prato")

(mostra-tokens (tokeniza omelete))